In [22]:
# [CELL 1] IMPORT REQUIRED LIBRARIES

import csv
import json
import shutil
from pathlib import Path

In [23]:
# [CELL 2] STUDENT-SPECIFIC CONFIGURATION

LAST_NAME = "Correa"
STUDENT_ID = "TUPM-26-1106"
SEED_DIGIT = 6
ID_SUM = 16

TARGET_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
TARGET_BEHAVIOR_CLASS = (SEED_DIGIT % 5) + 1

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Target Operating System:", TARGET_OS)
print("Target Behavior Class:", TARGET_BEHAVIOR_CLASS)

Student: Correa
Student ID: TUPM-26-1106
Seed Digit: 6
ID Sum: 16
Target Operating System: Android
Target Behavior Class: 2


In [24]:
# [CELL 3] CREATE DIRECTORIES AND DEFINE FILE PATHS

BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"

INPUT_FILE = DATA_DIR / "user_behavior_dataset.csv"
FILTERED_FILE = OUTPUT_DIR / "filtered_mobile_users.csv"
SUMMARY_FILE = OUTPUT_DIR / "mobile_usage_summary.json"
BACKUP_FILE = BACKUP_DIR / "mobile_users_backup.csv"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

print("Project Directory:", BASE_DIR)
print("Input File:", INPUT_FILE)
print("Input File Exists:", INPUT_FILE.exists())
print("Output Directory:", OUTPUT_DIR)
print("Backup Directory:", BACKUP_DIR)

Project Directory: /Users/jherielle/Downloads/CORREA/Lab6_File_Handling_Correa/Lab6_File_Handling_Additional_Exercises
Input File: /Users/jherielle/Downloads/CORREA/Lab6_File_Handling_Correa/Lab6_File_Handling_Additional_Exercises/data/user_behavior_dataset.csv
Input File Exists: True
Output Directory: /Users/jherielle/Downloads/CORREA/Lab6_File_Handling_Correa/Lab6_File_Handling_Additional_Exercises/output
Backup Directory: /Users/jherielle/Downloads/CORREA/Lab6_File_Handling_Correa/Lab6_File_Handling_Additional_Exercises/backup


In [25]:
# [CELL 4] INSPECT DATASET AND PREVIEW RECORDS

with open(INPUT_FILE, "r", encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)

    print("===== DATASET COLUMNS =====")
    print(reader.fieldnames)

    print("\n===== FIRST RECORD =====")
    first_record = next(reader, None)
    print(first_record)

===== DATASET COLUMNS =====
['User ID', 'Device Model', 'Operating System', 'App Usage Time (min/day)', 'Screen On Time (hours/day)', 'Battery Drain (mAh/day)', 'Number of Apps Installed', 'Data Usage (MB/day)', 'Age', 'Gender', 'User Behavior Class']

===== FIRST RECORD =====
{'User ID': '1', 'Device Model': 'Google Pixel 5', 'Operating System': 'Android', 'App Usage Time (min/day)': '393', 'Screen On Time (hours/day)': '6.4', 'Battery Drain (mAh/day)': '1872', 'Number of Apps Installed': '67', 'Data Usage (MB/day)': '1122', 'Age': '40', 'Gender': 'Male', 'User Behavior Class': '4'}


In [26]:
# [CELL 5] READ AND FILTER MOBILE DEVICE RECORDS

selected_users = []

total_records = 0
os_matching_records = 0
invalid_records = 0

with open(INPUT_FILE, "r", encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)

    columns = {name.strip().lower(): name for name in reader.fieldnames}

    required_columns = [
        "user id",
        "device model",
        "operating system",
        "battery drain (mah/day)",
        "data usage (mb/day)",
        "user behavior class"
    ]

    if not all(column in columns for column in required_columns):
        raise ValueError("Required dataset columns are missing.")

    for row in reader:
        total_records += 1

        os_name = (row[columns["operating system"]] or "").strip()

        if os_name.lower() != TARGET_OS.lower():
            continue

        os_matching_records += 1

        try:
            behavior_class = int(float(row[columns["user behavior class"]]))
            battery = float(row[columns["battery drain (mah/day)"]])
            data_usage = float(row[columns["data usage (mb/day)"]])

        except (ValueError, TypeError):
            invalid_records += 1
            continue

        if behavior_class == TARGET_BEHAVIOR_CLASS:
            selected_users.append({
                "User ID": row[columns["user id"]],
                "Device Model": row[columns["device model"]],
                "Operating System": os_name,
                "Battery Drain (mAh/day)": battery,
                "Data Usage (MB/day)": data_usage
            })

print("Total Records:", total_records)
print("Records Matching OS:", os_matching_records)
print("Records in Target Behavior Class:", len(selected_users))
print("Invalid Records:", invalid_records)

Total Records: 700
Records Matching OS: 554
Records in Target Behavior Class: 114
Invalid Records: 0


In [27]:
# [CELL 6] CALCULATE AVERAGE BATTERY DRAIN AND DATA USAGE

if selected_users:
    average_battery_drain = sum(
        user["Battery Drain (mAh/day)"] for user in selected_users
    ) / len(selected_users)

    average_data_usage = sum(
        user["Data Usage (MB/day)"] for user in selected_users
    ) / len(selected_users)

else:
    average_battery_drain = None
    average_data_usage = None

print("Average Battery Drain:", average_battery_drain, "mAh/day")
print("Average Data Usage:", average_data_usage, "MB/day")

Average Battery Drain: 887.1140350877193 mAh/day
Average Data Usage: 451.43859649122805 MB/day


In [28]:
# [CELL 7] EXPORT FILTERED RECORDS TO CSV

fieldnames = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)"
]

with open(FILTERED_FILE, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)

    writer.writeheader()
    writer.writerows(selected_users)

print("Filtered CSV created:", FILTERED_FILE.exists())
print("Exported Records:", len(selected_users))

Filtered CSV created: True
Exported Records: 114


In [29]:
# [CELL 8] CREATE JSON SUMMARY REPORT

summary = {
    "student": LAST_NAME,
    "student_id": STUDENT_ID,
    "seed_digit": SEED_DIGIT,
    "id_sum": ID_SUM,
    "operating_system": TARGET_OS,
    "target_behavior_class": TARGET_BEHAVIOR_CLASS,
    "total_records": total_records,
    "records_matching_os": os_matching_records,
    "records_in_target_behavior_class": len(selected_users),
    "invalid_records": invalid_records,
    "average_battery_drain_mAh_per_day": average_battery_drain,
    "average_data_usage_MB_per_day": average_data_usage
}

with open(SUMMARY_FILE, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON Summary created:", SUMMARY_FILE.exists())

JSON Summary created: True


In [30]:
# [CELL 9] VERIFY EXPORTED CSV ROW COUNT

with open(FILTERED_FILE, "r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    exported_count = sum(1 for row in reader)

count_verified = exported_count == len(selected_users)

print("Expected Records:", len(selected_users))
print("Exported Records:", exported_count)
print("Count Verified:", count_verified)

Expected Records: 114
Exported Records: 114
Count Verified: True


In [31]:
# [CELL 10] CREATE BACKUP AND CHECK FILE SIZE

shutil.copy2(FILTERED_FILE, BACKUP_FILE)

backup_exists = BACKUP_FILE.exists()

backup_size = BACKUP_FILE.stat().st_size if backup_exists else 0

backup_verified = (
    backup_exists
    and FILTERED_FILE.read_bytes() == BACKUP_FILE.read_bytes()
)

print("Backup Exists:", backup_exists)
print("Backup Verified:", backup_verified)
print("Backup File Size:", backup_size, "bytes")

Backup Exists: True
Backup Verified: True
Backup File Size: 4573 bytes


In [32]:
# [CELL 11] DISPLAY FINAL REPORT AND FILE VERIFICATION

with open(SUMMARY_FILE, "r", encoding="utf-8") as file:
    final_summary = json.load(file)

print("===== MOBILE DEVICE USAGE REPORT =====")
print(json.dumps(final_summary, indent=4))

print("\n===== FILE VERIFICATION =====")
print("Filtered CSV exists:", FILTERED_FILE.exists())
print("JSON Summary exists:", SUMMARY_FILE.exists())
print("Backup exists:", backup_exists)
print("Backup verified:", backup_verified)
print("Count verified:", count_verified)

===== MOBILE DEVICE USAGE REPORT =====
{
    "student": "Correa",
    "student_id": "TUPM-26-1106",
    "seed_digit": 6,
    "id_sum": 16,
    "operating_system": "Android",
    "target_behavior_class": 2,
    "total_records": 700,
    "records_matching_os": 554,
    "records_in_target_behavior_class": 114,
    "invalid_records": 0,
    "average_battery_drain_mAh_per_day": 887.1140350877193,
    "average_data_usage_MB_per_day": 451.43859649122805
}

===== FILE VERIFICATION =====
Filtered CSV exists: True
JSON Summary exists: True
Backup exists: True
Backup verified: True
Count verified: True
